# Lesson 03 Lab — PyTorch AMP: autocast and GradScaler

**Puzzle:** Can mixed-precision training be reduced to wrapping the forward pass in autocast?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

The AMP loop contains FP32 parameters and optimizer state, autocast-selected forward activations, gradients, a scalar loss scale, and an optimizer update. These objects do not all share one dtype or lifetime.

### Core mechanism

If `g` is the true gradient and `S` is the loss scale, backward first produces `S·g`; unscale restores `g` before clipping or the optimizer step. `GradScaler` skips the step when non-finite gradients are found and adapts `S`. Autocast independently chooses eligible forward-operation dtypes.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "03-pytorch-amp"
device = require_cuda()
torch.manual_seed(2026 + 3)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

BF16 often does not need scaling because of its exponent range, while FP16 can benefit from it. Scaling adds control logic and cannot repair a forward activation that already overflowed.

### What this code tests

The notebook prints parameter and output dtypes, runs the complete update loop, and records gradient finiteness rather than stopping after one autocast forward.

**Experiment:** Train a small CUDA MLP with BF16 autocast and GradScaler while recording loss, parameter dtype, output dtype, gradient finiteness, and scale history.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
model = torch.nn.Sequential(torch.nn.Linear(512, 1024), torch.nn.GELU(), torch.nn.Linear(1024, 64)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler("cuda")
x = torch.randn(256, 512, device=device); target = torch.randn(256, 64, device=device)
history = []
for step in range(6):
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        pred = model(x); loss = torch.nn.functional.mse_loss(pred, target)
    scaler.scale(loss).backward()
    finite = all(p.grad is None or torch.isfinite(p.grad).all().item() for p in model.parameters())
    scale_before = scaler.get_scale(); scaler.step(optimizer); scaler.update()
    history.append({"step": step, "loss": round(loss.item(), 7), "output_dtype": str(pred.dtype),
                    "grads_finite": bool(finite), "scale": float(scale_before)})
result = base_result(3, "pytorch-gpu")
result.update({"parameter_dtype": str(next(model.parameters()).dtype), "history": history,
               "conclusion": "The full autocast-scale-backward-step-update loop completed with finite gradients."})


## 3. Inspect the evidence

A valid loop needs finite gradients and an optimizer update. An autocast dtype printout alone is not a training result.

### Acceptance and rollback gate

Verify the order `zero_grad -> autocast forward -> scale(loss).backward -> unscale/step -> update`, record finite gradients and scale history, and keep the loss objective identical to the FP32 baseline.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "The full autocast-scale-backward-step-update loop completed with finite gradients.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:11+00:00",
  "history": [
    {
      "grads_finite": true,
      "loss": 1.0376294,
      "output_dtype": "torch.bfloat16",
      "scale": 65536.0,
      "step": 0
    },
    {
      "grads_finite": true,
      "loss": 0.9307048,
      "output_dtype": "torch.bfloat16",
      "scale": 65536.0,
      "step": 1
    },
    {
      "grads_finite": true,
      "loss": 0.836614,
      "output_dtype": "torch.bfloat16",
      "scale": 65536.0,
      "step": 2
    },
    {
      "grads_finite": true,
      "loss": 0.751017,
      "output_dtype": "torch.bfloat16",
      "scale": 65536.0,
      "step": 3
    },
    {
      "

## 4. Explain the result

AMP is a control loop across forward, backward, unscale, step, and update—not a global dtype switch.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).